# Week 14 — Cloud Applications and the Capstone

**Spatial Data Science with Python · MSc GIS / AGMS 1333**  
**Official Table 4 topic:** Cloud Computing · GeoJSON, priority layers, WebGIS

**Learning outcome.** Export an interpretable priority-review layer.

**What you will implement:** Quantiles, GeoJSON, optional folium mapping.  
**Mathematical model:** Flag=1 when LST≥75th percentile; else 0.

> **Data provenance:** The Urban Teaching Grid is a **synthetic** 12 × 12 grid at 250-m cell spacing. Its temperatures and facilities are simulated; do not present its numerical outputs as field measurements. All distance calculations in the labs use the projected coordinates EPSG:32647 where relevant.

**How to run.** In Colab, upload this notebook from *File → Upload notebook*. Run each cell top-to-bottom. The setup cell below will ask for `DATA_ONLY.zip` only if the data folder is absent. In JupyterLab, extract the complete ZIP and open this notebook in `01_Core_Labs/`.


## 1. Prepare the environment and locate the data

The same setup works in JupyterLab and Google Colab. It selects an existing local data directory, or prompts for the packaged dataset in Colab. `assert` fails early if the archive is incomplete.

In [ ]:
from pathlib import Path
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Local Jupyter: data/ or ../data. Colab: upload DATA_ONLY.zip once per runtime.
candidates = [Path('data'), Path('../data'), Path('/content/data')]
DATA = next((p for p in candidates if (p / 'cells.csv').is_file()), None)
if DATA is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Cannot find data/. Extract the course ZIP and keep the data folder with the notebooks.') from exc
    print('Select DATA_ONLY.zip from the downloaded course package:')
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.endswith('.zip')), None)
    if zip_name is None:
        raise FileNotFoundError('Please upload DATA_ONLY.zip and run this cell again.')
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
        archive.extractall(Path.cwd())
    DATA = Path('data')
assert (DATA / 'metadata.json').is_file(), 'Data package incomplete.'
print('Dataset folder:', DATA.resolve())
print('Synthetic teaching data: not field observations.')


## 2. Key concepts and expected data

The analysis follows **input → cleaning → computation → validation → interpretation**. Always inspect observation keys, coordinate units, missing values, and the intended study period before fitting a model.

**Mathematical foundation:** Flag=1 when LST≥75th percentile; else 0

**Method family:** Quantiles, GeoJSON, optional folium mapping.

## 3. Main computation

Run this cell after the setup. The block calculates the principal statistics or fits the weekly model. The original course code is retained so results remain aligned with the slide examples.

In [ ]:
import json
cells=pd.read_csv(DATA/'cells.csv');obs=pd.read_csv(DATA/'observations_clean.csv')
june=cells.merge(obs[obs.date=='2025-06-01'],on='cell_id',validate='one_to_one')
q75=june.lst_c.quantile(.75)
june['priority_review']=(june.lst_c>=q75).astype(int)
print('Teaching priority threshold:',round(q75,2),'°C');print('Flagged cells:',june.priority_review.sum())
print(june.groupby('sector')[['lst_c','priority_review']].agg({'lst_c':'mean','priority_review':'sum'}).round(2))

### Reading the main code

**Priority criteria.** Calculate the 75th percentile of the synthetic June LST scene and flag cells meeting the criterion.

- `pd.read_csv` — Reads CSV text into a pandas DataFrame. Inspect head() and column types after loading.
- `merge` — Joins tables on a declared key; validate cardinality to avoid duplicate matches.
- `quantile` — Returns a data percentile for thresholds or distribution summaries.
- `groupby` — Groups rows before counting or aggregating by sector, date, or other key.

**Verification questions:** Which variables contain the inputs? What new object does the cell create? What numerical result should be checked before continuing?

## 4. Map, plot, or evaluate the result

The final block turns results into a labelled graphic or output artifact. Keep units in axes/legends, and distinguish model predictions from observations.

In [ ]:
geo=json.loads((DATA/'cells.geojson').read_text())
lookup=june.set_index('cell_id')[['lst_c','priority_review']].to_dict('index')
for f in geo['features']:
    f['properties'].update({k:(float(v) if k=='lst_c' else int(v)) for k,v in lookup[f['properties']['cell_id']].items()})
out=Path('capstone_priority.geojson');out.write_text(json.dumps(geo))
print('Exported:',out.resolve())
try:
 import folium
 lon,lat=geo['features'][0]['geometry']['coordinates'][0][0]
 m=folium.Map(location=[lat,lon],zoom_start=13)
 folium.GeoJson(geo,name='Synthetic priority cells',style_function=lambda f:{'fillColor':'#d97706' if f['properties']['priority_review'] else '#0f766e','color':'#ffffff','weight':.5,'fillOpacity':.65}).add_to(m)
 m.save('capstone_webgis.html'); print('Map saved as capstone_webgis.html')
except ImportError:
 print('Optional interactive WebGIS: %pip install folium')

### Reading the output code

**Export.** Join flags by `cell_id`, update feature attributes in GeoJSON, and write the resulting file.

- `json.` — Reads or writes structured GeoJSON / STAC metadata.

**WebGIS.** Optional `folium` creates a map for communication. A static 2D map is a decision-support view, not automatically a digital twin.

## 5. Critical interpretation

### Interpret your results

Complete the following short report **after** running the notebook:

1. **Research question:** What geographic question does this notebook address?
2. **Observation unit and period:** What does one row/pixel represent? What is its date and CRS?
3. **Method:** Why is this method suitable, and what assumptions does it make?
4. **Results:** Report two numerical outputs or figure patterns with units.
5. **Validity:** State one data limitation and one model/evaluation limitation.
6. **Extension:** Modify one parameter, compare results, and describe what changed.


## 6. Questions and independent practice

1. Export a priority layer for a second synthetic date with preserved feature IDs and explain its limitations.
2. What would happen if the spatial unit, study date, or CRS changed?
3. Is this analysis descriptive, inferential, predictive, or a decision-support example? Justify your answer.
4. Which assumption should be tested before use in a real project?

**Next step:** Open the matching advanced notebook in `02_Advanced_Experiments/` for four short method variations.